In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [ ]:
import requests

url = "https://www.gutenberg.org/files/1524/1524-0.txt"
data = requests.get(url).text

print(data[:500])

*** START OF THE PROJECT GUTENBERG EBOOK 1524 ***




THE TRAGEDY OF HAMLET, PRINCE OF DENMARK

by William Shakespeare




Contents

 ACT I
 Scene I. Elsinore. A platform before the Castle
 Scene II. Elsinore. A room of state in the Castle
 Scene III. A room in Polonius’s house
 Scene IV. The platform
 Scene V. A more remote part of the Castle

 ACT II
 Scene I. A room in Polonius’s house
 Scene II. A room in the Castle

 ACT III
 Scene I. A room in the Castle
 Scene II. A hall in the Castle
 Sc


In [3]:
import re

# Lowercase the Hamlet text
text = data.lower()

# Remove unwanted characters from the Hamlet text
text = re.sub(r'[^a-zA-Z\s]', '', text)

# Split the Hamlet text into sentences (lines)
lines = text.split('\n')
lines = [line.strip() for line in lines if line.strip() != ""]

In [4]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(lines)

total_words = len(tokenizer.word_index) + 1
print("Total words:", total_words)

Total words: 4824


In [5]:
input_sequences = []

for line in lines:
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

In [6]:
max_len = max(len(seq) for seq in input_sequences)

input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_len, padding='pre')
)

In [7]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# One-hot encode output
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

In [8]:
# Build LSTM Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential()

# Embedding layer (IMPORTANT: input_length added back)
model.add(Embedding(input_dim=total_words, output_dim=100, input_length=max_len-1))

# LSTM layer
model.add(LSTM(150))

# Output layer
model.add(Dense(total_words, activation='softmax'))

# Compile model
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Show summary
model.build(input_shape=(None, max_len-1))
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 16, 100)        │       482,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 150)            │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4824)           │       728,424 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,361,424 (5.19 MB)

 Trainable params: 1,361,424 (5.19 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
history = model.fit(X, y, epochs=100, verbose=1)

Epoch 1/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.0390 - loss: 6.8326
Epoch 2/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.0525 - loss: 6.3779
Epoch 3/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.0657 - loss: 6.1252
Epoch 4/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.0837 - loss: 5.8555
Epoch 5/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.0995 - loss: 5.5646
Epoch 6/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.1128 - loss: 5.2779
Epoch 7/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.1261 - loss: 4.9985
Epoch 8/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.1395 - loss: 4.7326
Epoch 9/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.1594 - loss: 4.4703
Epoch 10/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.1860 - loss: 4.2135
Epoch 11/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.2209 - loss: 3.9593
Epoch 12/100
835/835 ━━━━━━━━━━━━━━━━

In [10]:
def predict_next_word(seed_text):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')

    predicted = np.argmax(model.predict(token_list), axis=-1)

    for word, index in tokenizer.word_index.items():
        if index == predicted:
            return word

In [21]:
print(predict_next_word("To be, or not to be: that is the")) # Prediction based on Hamlet's text
print(predict_next_word("There are more things in heaven and earth")) # Prediction based on Hamlet's text
print(predict_next_word("I am going to")) # Prediction based on Hamlet's text

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
question
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
horatio
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
this


In [22]:
model.save("next_word_model.h5")

In [23]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)